<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/seq2one/stage_07_02b_mlp_seq2one_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_02b - SEQ2ONE - MLP - Tuning**



# **BLOQUE DE EJECUCIÓN COMPLETO**

In [1]:
window_sizes = [180]
targets = ['delta_60']
splits = ['train', 'valid', 'test']

## **1. Imports + paths**

In [2]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [18]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

In [19]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [20]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl')}

## **4. Reproducibilidad**

In [21]:
#def set_seeds(seed: int = 42) -> None:
#    random.seed(seed)
#    np.random.seed(seed)
#    os.environ["PYTHONHASHSEED"] = str(seed)
#
#set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [22]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [23]:
#print(compute_seq2one_metrics.__doc__)

## **6. Carga de data windows**

In [24]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y


In [25]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [26]:
from typing import Any, Dict, Mapping
from pathlib import Path

# --------------------------------------------------
# Carga completa: ventanas + scaler por window_size y target
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path
    scalers_path[target] -> Path
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]
    scaler_path = scalers_paths[target]

    # --------------------------
    # 3) Carga
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    scaler = load_scaler(scaler_path)

    # --------------------------
    # 4) Inferir horizonte
    # --------------------------
    horizon = int(target.split("_")[-1])

    # --------------------------
    # 5) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [27]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [28]:
def create_bundles(window_size, targets: list, windows_paths=windows_paths, scalers_paths=scalers_paths, *, flatten_X=False):

    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
        )
        if flatten_X:
            b["train"]["X"] = maybe_flatten_X(b["train"]["X"], flatten=True)
            b["valid"]["X"] = maybe_flatten_X(b["valid"]["X"], flatten=True)
            b["test"]["X"]  = maybe_flatten_X(b["test"]["X"],  flatten=True)
        bundles.append(b)

    # prints (opcional)
    for b in bundles:
        print(f"H{b['horizon']} Train:", b["train"]["X"].shape, b["train"]["y"].shape)
        print(f"H{b['horizon']} Valid:", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print(f"H{b['horizon']} Test :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [29]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [30]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [31]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [32]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML**

In [33]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

In [45]:
def get_metrics_torch(bundle, model, *, device) -> tuple[dict, dict]:

    # -------- VALID --------
    X_valid = bundle["valid"]["X"]
    y_valid = np.asarray(bundle["valid"]["y"]).reshape(-1)
    y_pred_valid = np.asarray(
        predict_mlp(model, X_valid, device=device)
    ).reshape(-1)

    metrics_valid = compute_seq2one_metrics(
        y_valid, y_pred_valid, compute_r2=True
    )

    # -------- TEST --------
    X_test = bundle["test"]["X"]
    y_test = np.asarray(bundle["test"]["y"]).reshape(-1)
    y_pred_test = np.asarray(
        predict_mlp(model, X_test, device=device)
    ).reshape(-1)

    metrics_test = compute_seq2one_metrics(
        y_test, y_pred_test, compute_r2=True
    )

    return metrics_valid, metrics_test

## **9. Gestión de dataset de métricas**

In [35]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [36]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

In [37]:
import gc, torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gc.collect()
torch.cuda.empty_cache()

# **DEFINICIÓN DE MODELO**

## **10. Definición del modelo — placeholder**

### **10.1. Baseline MLP Configuration (SEQ2ONE – delta_60)**


**Hiperparámetros baseline**


| Hiperparámetro   | Valor   |
|------------------|---------|
| in_dim           | 1200    |
| hidden_dim       | 128     |
| dropout          | 0.0     |
| learning_rate    | 1e-3    |
| weight_decay     | 1e-4    |
| max_epochs       | 30      |
| patience         | 5       |
| optimizer        | Adam    |
| loss_function    | MSE     |

In [68]:
import pandas as pd
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

df_mlp_all_sizes = load_seq2one_metrics_if_exists(name="mlp")
df_mlp_delta_60 = df_mlp_all_sizes[df_mlp_all_sizes["target"] == "delta_60"].copy()

In [69]:
df_mlp_delta_60

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,hidden_dim,dropout,lr,w_decay,batch_size,max_epochs,patience
0,mlp,test,30,delta_60,60,48.197694,75.094125,0.158334,0.612614,128,0.0,0.001,0.0001,16384,30,5
1,mlp,valid,30,delta_60,60,30.792758,44.250947,0.189710,0.635432,128,0.0,0.001,0.0001,16384,30,5
8,mlp,test,60,delta_60,60,41.986391,67.616881,0.323348,0.696595,128,0.0,0.001,0.0001,16384,30,5
9,mlp,valid,60,delta_60,60,25.739746,39.123824,0.370360,0.715509,128,0.0,0.001,0.0001,16384,30,5
16,mlp,test,90,delta_60,60,39.345348,65.865070,0.354628,0.704880,128,0.0,0.001,0.0001,16384,30,5
17,mlp,valid,90,delta_60,60,23.108619,37.027374,0.422718,0.738037,128,0.0,0.001,0.0001,16384,30,5
24,mlp,test,120,delta_60,60,38.710646,64.403748,0.380758,0.709230,128,0.0,0.001,0.0001,16384,30,5
25,mlp,valid,120,delta_60,60,23.325614,37.396104,0.395885,0.733588,128,0.0,0.001,0.0001,16384,30,5
32,mlp,test,180,delta_60,60,36.526562,61.421779,0.447396,0.731242,128,0.0,0.001,0.0001,16384,30,5
33,mlp,valid,180,delta_60,60,21.795650,35.331207,0.450572,0.744949,128,0.0,0.001,0.0001,16384,30,5


### **11.2. Imports (PyTorch) + semillas**

In [70]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [71]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # reproducibilidad (puede bajar performance, pero estable)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


### **11.2. Dataset/DataLoader desde bundle**

In [72]:
def make_loaders_from_bundle(bundle, *, batch_size: int = 4096, num_workers: int = 0) -> dict:
    loaders = {}
    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 2:
            raise ValueError(f"[{split}] X debe ser 2D (n, d). Got shape={X.shape}")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"[{split}] X y y deben tener mismo n. X={X.shape}, y={y.shape}")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )
    return loaders

In [73]:
#loaders_60 = make_loaders_from_bundle(bundle_60, batch_size=16384)
#loaders_90 = make_loaders_from_bundle(bundle_90, batch_size=16384)

### **11.3. Definición del modelo MLP (simple y controlado)**

In [74]:
import torch
import torch.nn as nn

def _get_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "gelu":
        return nn.GELU()
    if name == "silu" or name == "swish":
        return nn.SiLU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"activation no soportada: {name}")

class MLPSeq2One(nn.Module):
    """
    MLP configurable para seq2one:
      - in_dim: dimensión de entrada (d)
      - hidden_dims: lista de dimensiones ocultas, p.ej. [256, 256, 128]
      - activation: relu | gelu | silu | tanh
      - dropout: float
      - norm: none | layernorm | batchnorm
      - residual: aplica skip en bloques con misma dim (opcional)
    """
    def __init__(
        self,
        *,
        in_dim: int,
        hidden_dims: list[int],
        activation: str = "relu",
        dropout: float = 0.0,
        norm: str = "none",
        residual: bool = False,
    ):
        super().__init__()
        if not hidden_dims:
            raise ValueError("hidden_dims debe tener al menos 1 capa")

        act = _get_activation(activation)
        norm = norm.lower()

        dims = [in_dim] + list(hidden_dims)
        layers: list[nn.Module] = []

        for i in range(len(dims) - 1):
            d_in, d_out = dims[i], dims[i + 1]

            block: list[nn.Module] = [nn.Linear(d_in, d_out)]

            if norm == "layernorm":
                block.append(nn.LayerNorm(d_out))
            elif norm == "batchnorm":
                block.append(nn.BatchNorm1d(d_out))
            elif norm == "none":
                pass
            else:
                raise ValueError(f"norm no soportada: {norm}")

            block.append(act)
            if dropout and dropout > 0:
                block.append(nn.Dropout(dropout))

            layers.append(nn.Sequential(*block))

        self.blocks = nn.ModuleList(layers)
        self.residual = residual
        self.out = nn.Linear(dims[-1], 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # asegurar 2D: (n, d)
        if x.ndim != 2:
            x = x.view(x.size(0), -1)

        h = x
        for block in self.blocks:
            h_new = block(h)
            # residual sólo si misma dimensión
            if self.residual and (h_new.shape[-1] == h.shape[-1]):
                h = h + h_new
            else:
                h = h_new

        return self.out(h)

### **11.4. Entrenamiento con early stopping (VALID)**

In [75]:
@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    sse = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        sse += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return sse / max(n, 1)

def train_mlp(
    loaders: dict,
    *,
    model_cfg: dict,      # <-- arquitectura
    optim_cfg: dict,      # <-- hiperparams aprendizaje
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,
):
    """
    Entrena MLP seq2one con early stopping en VALID por MSE.
    Retorna: (model_best, best_valid_mse, epochs_ran)
    """
    if verbose:
        print(
            f"[{_ts()}] [TRAIN] START | model_cfg={model_cfg} | "
            f"optim_cfg={optim_cfg} | max_epochs={max_epochs} patience={patience} | device={device.type}"
        )

    t_global = time.perf_counter()

    model = MLPSeq2One(**model_cfg).to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=float(optim_cfg.get("lr", 1e-3)),
        weight_decay=float(optim_cfg.get("weight_decay", 0.0)),
    )
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0
    epochs_ran = 0

    for epoch in range(1, max_epochs + 1):
        epochs_ran = epoch
        t_epoch = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={train_loss_sum/max(train_n,1):.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        t0 = time.perf_counter()
        valid_mse = evaluate_mse(model, loaders["valid"], device)
        dt_valid = time.perf_counter() - t0
        dt_epoch = time.perf_counter() - t_epoch

        improved = valid_mse < (best_valid - 1e-9)
        if improved:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(
                f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                f"valid_mse={valid_mse:.6f} | {flag} | dt_valid={dt_valid:.2f}s | dt_epoch={dt_epoch:.2f}s"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | best_valid_mse={best_valid:.6f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} | epochs_ran={epochs_ran} | dt_total={dt_all:.2f}s")

    return model, best_valid, epochs_ran

### **11.5. Predicciones MLP**


In [76]:
@torch.no_grad()
def predict_mlp(model, X: np.ndarray, *, device: torch.device, batch_size: int = 32768) -> np.ndarray:
    """
    Predice con un modelo MLP PyTorch en batches.
    Retorna shape (n_samples,)
    """
    model.eval()

    X = np.asarray(X, dtype=np.float32)

    # 🔎 Validación crítica de shape
    if X.ndim != 2:
        raise ValueError(f"X debe ser 2D (n, d). Got shape={X.shape}")

    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb).squeeze(-1)
        preds.append(yb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)

## **12. Ejecución completa**

In [77]:
def run_mlp(
    window_size: int,
    *,
    seed: int,
    model_cfg: dict,
    optim_cfg: dict,
    train_cfg: dict,
    verbose: bool = True,
):
    set_seed(seed)

    size = window_size
    n_features = int(train_cfg.get("n_features", 36))

    if verbose:
        print("\n" + "=" * 80)
        print(
            f"[{_ts()}] MLP | SEQ2ONE | seed={seed} | L{size} | in_dim={size*n_features}\n"
            f"model_cfg={model_cfg}\noptim_cfg={optim_cfg}\ntrain_cfg={train_cfg}"
        )
        print("=" * 80)

    targets = train_cfg.get("targets", ["delta_60", "delta_90", "ret_60", "ret_90"])
    rows = []
    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        t_target = time.perf_counter()
        if verbose:
            print(f"\n[{_ts()}] [{i}/{len(targets)}] START target='{target}' | L{size}")

        # BUILD BUNDLE
        (bundle,) = create_bundles(
            window_size=size,
            targets=[target],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,
        )

        # LOADERS
        loaders = make_loaders_from_bundle(
            bundle,
            batch_size=int(train_cfg.get("batch_size", 16384)),
            num_workers=int(train_cfg.get("num_workers", 0)),
        )

        # TRAIN
        _model_cfg = dict(model_cfg)
        _model_cfg["in_dim"] = size * n_features  # forzar consistencia con L y n_features

        model, best_valid_mse, epochs_ran = train_mlp(
            loaders,
            model_cfg=_model_cfg,
            optim_cfg=optim_cfg,
            max_epochs=int(train_cfg.get("max_epochs", 30)),
            patience=int(train_cfg.get("patience", 5)),
            device=device,
            verbose=verbose,
            log_every=int(train_cfg.get("log_every", 0)),
        )

        # liberar TRAIN
        del bundle["train"]
        gc.collect()

        # METRICS
        metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)

        # DF APPEND (+ registrar seed y best_valid_mse)
        df_va = metrics_to_df(
            metrics_valid, model="mlp", split="valid",
            horizon=bundle["horizon"], window_size=bundle["window_size"], target=bundle["target"],
        )
        df_te = metrics_to_df(
            metrics_test, model="mlp", split="test",
            horizon=bundle["horizon"], window_size=bundle["window_size"], target=bundle["target"],
        )

        for df_ in (df_va, df_te):
            df_["seed"] = seed
            df_["best_valid_mse"] = best_valid_mse
            df_["epochs_ran"] = epochs_ran
            # opcional: guardar config serializada
            df_["model_cfg"] = str(_model_cfg)
            df_["optim_cfg"] = str(optim_cfg)

        rows.append(df_va)
        rows.append(df_te)

        # CLEANUP
        del bundle, model, metrics_valid, metrics_test, loaders, df_va, df_te
        gc.collect()

        if verbose:
            print(f"[{_ts()}] [{i}/{len(targets)}] DONE target='{target}' | dt_total={time.perf_counter()-t_target:.2f}s")

    df_mlp_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model", "seed"])
          .reset_index(drop=True)
    )

    if verbose:
        print(f"\n[{_ts()}] [FINAL] OK | rows={len(df_mlp_metrics)} | dt_total={time.perf_counter()-t_global:.2f}s")

    return df_mlp_metrics

In [78]:
import pandas as pd
import gc

def run_mlp_incremental(
    window_sizes: list[int],
    *,
    seeds: list[int],
    configs: list[dict],   # cada item: {"cfg_id": "...", "model_cfg": {...}, "optim_cfg": {...}, "train_cfg": {...}}
    name: str = "mlp",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    if df_all.empty:
        df_all = pd.DataFrame(columns=[
            "cfg_id","seed","model","split","window_size","target","horizon_min",
            "MAE","RMSE","R2","DA",
            "best_valid_mse","epochs_ran",
        ])

    # llaves únicas por corrida
    key_cols = ["cfg_id","seed","model","window_size","target","split","horizon_min"]

    for ws in window_sizes:
        for cfg in configs:
            cfg_id = cfg["cfg_id"]

            for seed in seeds:
                # verificar si ya existe TODO para este (ws, cfg_id, seed)
                df_exist = df_all[
                    (df_all["cfg_id"] == cfg_id) &
                    (df_all["seed"] == seed) &
                    (df_all["window_size"] == ws) &
                    (df_all["model"] == "mlp")
                ]

                # esperado: 4 targets x 2 splits = 8 filas POR seed y cfg
                if len(df_exist) >= 8:
                    if verbose:
                        print(f"[SKIP] L{ws} cfg={cfg_id} seed={seed}: ya hay {len(df_exist)} filas.")
                    continue

                if verbose:
                    print("\n" + "="*90)
                    print(f"[RUN] MLP incremental | L{ws} | cfg={cfg_id} | seed={seed}")
                    print("="*90)

                df_new = run_mlp(
                    ws,
                    seed=seed,
                    model_cfg=cfg["model_cfg"],
                    optim_cfg=cfg["optim_cfg"],
                    train_cfg=cfg["train_cfg"],
                    verbose=verbose,
                ).copy()

                df_new["cfg_id"] = cfg_id
                df_new["seed"] = seed

                # anti-duplicados
                existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
                mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
                df_new = df_new.loc[mask_keep].copy()

                if df_new.empty:
                    if verbose:
                        print(f"[INFO] L{ws} cfg={cfg_id} seed={seed}: no había filas nuevas.")
                    continue

                df_all = pd.concat([df_all, df_new], ignore_index=True)
                df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

                save_seq2one_metrics(df_all, name=name, base_dir=base_dir)
                if verbose:
                    print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

                del df_new
                gc.collect()

    return df_all

In [79]:
#df_mlp_all_sizes = load_seq2one_metrics_if_exists(name="mlp", base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics")

In [80]:
MLP_ALL_TRAIN = '''
df_mlp_all_sizes = run_mlp_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="mlp",   # genera seq2one_ridge_metrics.parquet
    verbose=True,
)
'''

# **TUNEO**

## **13. Tuning de Arquitectura (capacidad del modelo)**

Afecta el bias–variance trade-off estructural.

Parámetros relevantes:
- hidden_dim
- n_layers (si agregas más capas)
- dropout
- Activación (ReLU, GELU, etc.)

Esto controla qué tan complejo puede ser el mapeo no lineal entre la ventana y delta_60.

### **13.1. Funciones para tuning de arquitectura**

In [92]:
import torch
import torch.nn as nn
import torch
import torch.nn as nn


class MLPSeq2One(nn.Module):
    def __init__(
        self,
        *,
        in_dim: int = 1200,
        hidden_dims: list[int] | None = None,
        dropout: float = 0.0,
        activation: str = "relu",
    ):
        super().__init__()

        if hidden_dims is None:
            hidden_dims = [128]

        if not hidden_dims or any(h <= 0 for h in hidden_dims):
            raise ValueError(f"hidden_dims inválido: {hidden_dims}")

        act_name = activation.lower()
        if act_name == "relu":
            Act = nn.ReLU
        elif act_name == "gelu":
            Act = nn.GELU
        else:
            raise ValueError(f"activation no soportada: {activation} (use 'relu' o 'gelu')")

        layers: list[nn.Module] = []
        prev = in_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(Act())
            if dropout and dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2:
            x = x.view(x.size(0), -1)
        return self.net(x)


In [93]:
ARCH_GRID = [
    # 1 capa
    {"hidden_dims": [64],  "dropout": 0.0,  "activation": "relu"},
    {"hidden_dims": [128], "dropout": 0.0,  "activation": "relu"},
    {"hidden_dims": [256], "dropout": 0.0,  "activation": "relu"},
    {"hidden_dims": [512], "dropout": 0.0,  "activation": "relu"},

    # 1 capa + dropout
    {"hidden_dims": [128], "dropout": 0.1,  "activation": "relu"},
    {"hidden_dims": [256], "dropout": 0.1,  "activation": "relu"},
    {"hidden_dims": [256], "dropout": 0.2,  "activation": "relu"},
    {"hidden_dims": [512], "dropout": 0.2,  "activation": "relu"},

    # 2 capas
    {"hidden_dims": [128, 64],   "dropout": 0.0, "activation": "relu"},
    {"hidden_dims": [256, 128],  "dropout": 0.0, "activation": "relu"},
    {"hidden_dims": [256, 128],  "dropout": 0.1, "activation": "relu"},
    {"hidden_dims": [256, 128],  "dropout": 0.2, "activation": "relu"},

    # 3 capas
    {"hidden_dims": [128, 128, 64],     "dropout": 0.1, "activation": "relu"},
    {"hidden_dims": [256, 256, 128],    "dropout": 0.1, "activation": "relu"},
    {"hidden_dims": [512, 256, 128],    "dropout": 0.2, "activation": "relu"},

    # GELU
    {"hidden_dims": [256],              "dropout": 0.1, "activation": "gelu"},
    {"hidden_dims": [256, 128],         "dropout": 0.1, "activation": "gelu"},
]

In [94]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ============================================================
# EVAL: MSE (VALID) para early stopping
# ============================================================
@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)


def _ts():
    return time.strftime("%H:%M:%S")


# ============================================================
# TRAIN: MLP seq2one + early stopping en VALID (por MSE)
# ============================================================
def train_mlp(
    loaders: dict,
    *,
    # ---- arquitectura ----
    in_dim: int = 1200,
    hidden_dims: list[int] | None = None,   # ej: [256] o [256,128]
    dropout: float = 0.0,
    activation: str = "relu",               # "relu" o "gelu"

    # ---- entrenamiento ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,  # 0 => no log por batch; si pones 50, log cada 50 batches
):
    """
    Entrena un MLP seq2one usando TRAIN y early stopping en VALID (por MSE).
    Logs: inicio/fin, train_loss, valid_mse, mejoras, early-stopping.

    Nota:
    - hidden_dims controla profundidad/anchura. Si None, usa [128] por compatibilidad.
    """
    if hidden_dims is None:
        hidden_dims = [128]  # baseline equivalente a hidden_dim=128

    if verbose:
        print(
            f"[{_ts()}] [TRAIN] START | in_dim={in_dim} hidden_dims={hidden_dims} "
            f"dropout={dropout} activation={activation} lr={lr} wd={weight_decay} "
            f"max_epochs={max_epochs} patience={patience} device={device.type}"
        )

    t_global = time.perf_counter()

    # ---- modelo (arquitectura tuneable) ----
    model = MLPSeq2One(
        in_dim=in_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
        activation=activation,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0
    epochs_ran = 0 ###

    for epoch in range(1, max_epochs + 1):
        epochs_ran = epoch ###
        t_epoch = time.perf_counter()

        # -------------------------
        # TRAIN EPOCH
        # -------------------------
        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                train_loss_avg_so_far = train_loss_sum / max(train_n, 1)
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={train_loss_avg_so_far:.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        # -------------------------
        # VALIDATION (MSE)
        # -------------------------
        t0 = time.perf_counter()
        valid_mse = evaluate_mse(model, loaders["valid"], device)
        dt_valid = time.perf_counter() - t0
        dt_epoch = time.perf_counter() - t_epoch

        # -------------------------
        # EARLY STOPPING
        # -------------------------
        improved = valid_mse < (best_valid - 1e-9)
        if improved:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(
                f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                f"valid_mse={valid_mse:.6f} | {flag} | dt_valid={dt_valid:.2f}s | dt_epoch={dt_epoch:.2f}s"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | patience={patience} | best_valid_mse={best_valid:.6f}")
            break

    if best_state is not None:
        if verbose:
            print(f"[{_ts()}] [TRAIN] Cargando best_state (best_valid_mse={best_valid:.6f}) ...")
        model.load_state_dict(best_state)

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} | dt_total={dt_all:.2f}s")

    return model, best_valid, epochs_ran


In [95]:
#cfg = {"hidden_dims": [256, 128], "dropout": 0.1, "activation": "relu"}
#model = train_mlp(loaders, device=device, in_dim=1200, **cfg)

In [96]:
@torch.no_grad()
def predict_mlp(
    model,
    X: np.ndarray,
    *,
    device: torch.device,
    batch_size: int = 32768
) -> np.ndarray:
    model.eval()

    X = np.asarray(X, dtype=np.float32)

    # 🔎 Validación crítica
    if X.ndim != 2:
        raise ValueError(f"X debe ser 2D (n, d). Got shape={X.shape}")

    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb_np = np.ascontiguousarray(X[i : i + batch_size])
        xb = torch.from_numpy(xb_np).to(device, non_blocking=True)

        yb = model(xb).squeeze(-1)
        preds.append(yb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)


In [97]:
# ============================================================
# RUN MLP (solo delta_60 + arquitectura tuneable) - AJUSTADO
# (tuneo grueso -> tuneo fino -> multi-seed)
# ============================================================
import pandas as pd
import gc
import time

def _ts():
    return time.strftime("%H:%M:%S")


def run_mlp_delta_60(
    window_size: int,
    *,
    seed: int,
    hidden_dims: list[int] | None = None,
    dropout: float = 0.0,
    activation: str = "relu",
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    batch_size: int = 16384,
    max_epochs: int = 30,
    patience: int = 5,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Runner para MLP seq2one en target=delta_60.
    Devuelve DF con métricas valid/test y tracking de:
      - arquitectura (hidden_dims, dropout, activation)
      - entrenamiento (lr, weight_decay, best_valid_mse, epochs_ran)
      - seed
    """
    # reproducibilidad por corrida
    set_seed(seed)

    size = window_size
    n_features = 36
    target = "delta_60"

    if hidden_dims is None:
        hidden_dims = [128]

    if verbose:
        print("\n" + "=" * 90)
        print(
            f"[{_ts()}] MLP | SEQ2ONE | seed={seed} | target={target} | "
            f"L{size} | in_dim={size*n_features} | hidden_dims={hidden_dims} | "
            f"dropout={dropout} | act={activation} | lr={lr} | w_decay={weight_decay} | "
            f"batch={batch_size} | max_epochs={max_epochs} | patience={patience}"
        )
        print("=" * 90)

    rows = []
    t_global = time.perf_counter()

    # -------------------------
    # BUILD BUNDLE
    # -------------------------
    if verbose:
        print(f"[{_ts()}]   [BUILD] Creando bundle (flatten_X=True) ...")
    t0 = time.perf_counter()

    (bundle,) = create_bundles(
        window_size=size,
        targets=[target],
        windows_paths=windows_paths,
        scalers_paths=scalers_paths,
        flatten_X=True,  # MLP necesita 2D
    )

    if verbose:
        dt = time.perf_counter() - t0
        try:
            xshape = bundle["train"]["X"].shape
            yshape = bundle["train"]["y"].shape
            print(f"[{_ts()}]   [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
        except Exception:
            print(f"[{_ts()}]   [BUILD] OK | dt={dt:.2f}s")

    # -------------------------
    # LOADERS
    # -------------------------
    if verbose:
        print(f"[{_ts()}]   [LOADERS] Creando DataLoaders ...")
    t0 = time.perf_counter()

    loaders = make_loaders_from_bundle(
        bundle,
        batch_size=batch_size,
    )

    if verbose:
        dt = time.perf_counter() - t0
        try:
            ntr = len(loaders["train"].dataset)
            nva = len(loaders["valid"].dataset)
            nte = len(loaders["test"].dataset)
            print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
        except Exception:
            print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

    # -------------------------
    # TRAIN
    # -------------------------
    if verbose:
        print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
    t0 = time.perf_counter()

    model, best_valid_mse, epochs_ran = train_mlp(
        loaders,
        in_dim=size * n_features,
        hidden_dims=hidden_dims,
        dropout=dropout,
        activation=activation,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        patience=patience,
        device=device,
        verbose=verbose,
    )

    if verbose:
        dt = time.perf_counter() - t0
        print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s | best_valid_mse={best_valid_mse:.6f} | epochs_ran={epochs_ran}")

    # liberar TRAIN (opcional)
    if verbose:
        print(f"[{_ts()}]   [MEM] Liberando bundle['train'] y gc.collect() ...")
    del bundle["train"]
    gc.collect()

    # -------------------------
    # PRED + METRICS
    # -------------------------
    if verbose:
        print(f"[{_ts()}]   [PRED] Predicciones + métricas (valid/test) ...")
    t0 = time.perf_counter()

    metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)

    if verbose:
        dt = time.perf_counter() - t0
        print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

    # -------------------------
    # DF APPEND
    # -------------------------
    rows.append(metrics_to_df(
        metrics_valid,
        model="mlp",
        split="valid",
        horizon=bundle["horizon"],
        window_size=bundle["window_size"],
        target=bundle["target"],
    ))

    rows.append(metrics_to_df(
        metrics_test,
        model="mlp",
        split="test",
        horizon=bundle["horizon"],
        window_size=bundle["window_size"],
        target=bundle["target"],
    ))

    df_mlp_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    # ---- tracking ----
    df_mlp_metrics["seed"] = seed
    df_mlp_metrics["hidden_dims"] = str(hidden_dims)
    df_mlp_metrics["dropout"] = float(dropout)
    df_mlp_metrics["activation"] = str(activation)
    df_mlp_metrics["lr"] = float(lr)
    df_mlp_metrics["w_decay"] = float(weight_decay)
    df_mlp_metrics["best_valid_mse"] = float(best_valid_mse)
    df_mlp_metrics["epochs_ran"] = int(epochs_ran)

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [DONE] rows={len(df_mlp_metrics)} | dt_total={dt_all:.2f}s")

    # cleanup
    del bundle, model, metrics_valid, metrics_test, loaders
    gc.collect()

    return df_mlp_metrics


In [98]:
# ============================================================
# RUN MLP INCREMENTAL (solo delta_60 + arquitectura tuneable) - AJUSTADO
#  - libera memoria
#  - guarda checkpoints por ws
#  - incluye seed + best_valid_mse + epochs_ran
# ============================================================
import pandas as pd
import gc
import torch

def run_mlp_incremental_delta_60(
    window_sizes: list[int],
    *,
    seed: int,
    hidden_dims: list[int] | None = None,
    dropout: float = 0.0,
    activation: str = "relu",
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    name: str = "mlp_arch_tuning_delta_60",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    if hidden_dims is None:
        hidden_dims = [128]

    # 1) Cargar si existe
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    # 2) Esquema mínimo esperado (ahora guardamos arquitectura + seed + training stats)
    base_cols = [
        "model","split","window_size","target","horizon_min",
        "MAE","RMSE","R2","DA",
        "seed","hidden_dims","dropout","activation","lr","w_decay",
        "best_valid_mse","epochs_ran",
    ]
    if df_all.empty:
        df_all = pd.DataFrame(columns=base_cols)
    else:
        for c in base_cols:
            if c not in df_all.columns:
                df_all[c] = pd.NA

    # 3) Normalizar tipos
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")
        df_all["seed"] = pd.to_numeric(df_all["seed"], errors="coerce").astype("Int64")

    # 4) Keys únicas por config + ws + split/horizon (+ seed)
    hidden_dims_str = str(hidden_dims)
    key_cols = [
        "model","window_size","target","split","horizon_min",
        "seed","hidden_dims","dropout","activation","lr","w_decay",
    ]

    # precomputar set de keys existentes (lo iremos actualizando)
    existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)

    # 5) Loop por window_size
    for ws in window_sizes:

        df_ws = df_all[
            (df_all["model"] == "mlp") &
            (df_all["window_size"] == ws) &
            (df_all["target"] == "delta_60") &
            (df_all["seed"] == seed) &
            (df_all["hidden_dims"] == hidden_dims_str) &
            (df_all["dropout"] == dropout) &
            (df_all["activation"] == activation) &
            (df_all["lr"] == lr) &
            (df_all["w_decay"] == weight_decay)
        ]

        # Esperadas: delta_60 x 2 splits (valid/test) = 2 filas (por seed y config)
        if len(df_ws) >= 2:
            if verbose:
                print(f"[SKIP] L{ws}: ya hay {len(df_ws)} filas para esta config (delta_60, seed={seed}).")
            continue

        if verbose:
            print("\n" + "="*100)
            print(
                f"[RUN] MLP incremental | target=delta_60 | seed={seed} | L{ws} | in_dim={ws*36} | "
                f"hidden_dims={hidden_dims} | dropout={dropout} | act={activation} | "
                f"lr={lr} | w_decay={weight_decay}"
            )
            print("="*100)

        # Entrenar/evaluar para ESTE ws (solo delta_60)
        df_new = run_mlp_delta_60(
            ws,
            seed=seed,
            hidden_dims=hidden_dims,
            dropout=dropout,
            activation=activation,
            lr=lr,
            weight_decay=weight_decay,
            verbose=verbose,
        ).copy()

        # Anti-duplicados por key (usando existing_keys precomputado)
        new_keys = [tuple(row) for row in df_new[key_cols].values]
        mask_keep = [k not in existing_keys for k in new_keys]
        df_new = df_new.loc[mask_keep].copy()

        if df_new.empty:
            if verbose:
                print(f"[INFO] L{ws}: no había filas nuevas para agregar.")
        else:
            # Merge + dedupe
            df_all = pd.concat([df_all, df_new], ignore_index=True)
            df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

            # actualizar existing_keys con lo agregado
            for k in new_keys:
                existing_keys.add(k)

            # Guardar checkpoint (paso a paso)
            save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

            if verbose:
                print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

        # 6) Liberar memoria por iteración
        del df_new
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return df_all

### **13.2. Ejecución de tuning**

In [99]:
ARCH_GRID = [
    # 1 capa
    {"hidden_dims": [64],  "dropout": 0.0,  "activation": "relu"},
    {"hidden_dims": [128], "dropout": 0.0,  "activation": "relu"},
    {"hidden_dims": [256], "dropout": 0.0,  "activation": "relu"},
    {"hidden_dims": [512], "dropout": 0.0,  "activation": "relu"},

    # 1 capa + dropout
    {"hidden_dims": [128], "dropout": 0.1,  "activation": "relu"},
    {"hidden_dims": [256], "dropout": 0.1,  "activation": "relu"},
    {"hidden_dims": [256], "dropout": 0.2,  "activation": "relu"},
    {"hidden_dims": [512], "dropout": 0.2,  "activation": "relu"},

    # 2 capas
    {"hidden_dims": [128, 64],   "dropout": 0.0, "activation": "relu"},
    {"hidden_dims": [256, 128],  "dropout": 0.0, "activation": "relu"},
    {"hidden_dims": [256, 128],  "dropout": 0.1, "activation": "relu"},
    {"hidden_dims": [256, 128],  "dropout": 0.2, "activation": "relu"},

    # 3 capas
    {"hidden_dims": [128, 128, 64],     "dropout": 0.1, "activation": "relu"},
    {"hidden_dims": [256, 256, 128],    "dropout": 0.1, "activation": "relu"},
    {"hidden_dims": [512, 256, 128],    "dropout": 0.2, "activation": "relu"},

    # GELU
    {"hidden_dims": [256],              "dropout": 0.1, "activation": "gelu"},
    {"hidden_dims": [256, 128],         "dropout": 0.1, "activation": "gelu"},
]


In [100]:
# ============================================================
# TUNING RUNNER: recorre ARCH_GRID y ejecuta incremental por config
# ============================================================
import time
import pandas as pd

def _ts():
    return time.strftime("%H:%M:%S")


def _arch_tag(hidden_dims: list[int], dropout: float, activation: str) -> str:
    # tag corto y "filename-friendly"
    hd = "-".join(map(str, hidden_dims))
    do = str(dropout).replace(".", "p")
    act = str(activation)
    return f"hd{hd}_do{do}_act{act}"


def run_mlp_arch_tuning_delta_60(
    window_sizes: list[int],
    *,
    seed: int,
    arch_grid: list[dict],
    lr: float = 1e-3,                 # fijo en tuning de arquitectura
    weight_decay: float = 1e-4,       # fijo en tuning de arquitectura
    base_name: str = "mlp_arch_tuning_delta_60",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Recorre un grid de arquitecturas y ejecuta incremental por config.
    Guarda checkpoints por arquitectura en un parquet distinto (name único).
    Retorna el último df_all (última arquitectura procesada).
    """
    last_df = pd.DataFrame()

    for k, cfg in enumerate(arch_grid, start=1):
        hidden_dims = cfg["hidden_dims"]
        dropout = float(cfg.get("dropout", 0.0))
        activation = cfg.get("activation", "relu")

        arch_tag = _arch_tag(hidden_dims, dropout, activation)
        name = f"{base_name}__{arch_tag}"

        if verbose:
            print("\n" + "=" * 110)
            print(f"[{_ts()}] [{k}/{len(arch_grid)}] ARCH TUNING | seed={seed} | {arch_tag} | lr={lr} | w_decay={weight_decay}")
            print("=" * 110)

        last_df = run_mlp_incremental_delta_60(
            window_sizes,
            seed=seed,
            hidden_dims=hidden_dims,
            dropout=dropout,
            activation=activation,
            lr=lr,
            weight_decay=weight_decay,
            name=name,
            base_dir=base_dir,
            verbose=verbose,
        )

    return last_df

In [101]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [102]:
import pandas as pd
import gc
import torch

def run_mlp_incremental_delta_60_onefile(
    window_sizes: list[int],
    arch_grid: list[dict],
    *,
    seed: int,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    name: str = "mlp_arch_tuning_delta_60_all",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    # 1) Cargar si existe
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    # 2) Esquema esperado (incluye seed + stats training)
    cols = [
        "model","split","window_size","target","horizon_min",
        "MAE","RMSE","R2","DA",
        "seed","hidden_dims","dropout","activation","lr","w_decay",
        "best_valid_mse","epochs_ran",
    ]
    if df_all.empty:
        df_all = pd.DataFrame(columns=cols)
    else:
        for c in cols:
            if c not in df_all.columns:
                df_all[c] = pd.NA

    # 3) Normalizar tipos
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")
        df_all["seed"] = pd.to_numeric(df_all["seed"], errors="coerce").astype("Int64")

    # 4) Key única por corrida
    key_cols = [
        "model","window_size","target","split","horizon_min",
        "seed","hidden_dims","dropout","activation","lr","w_decay",
    ]

    # set incremental para anti-duplicados
    existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)

    # 5) Loop por arquitectura y window_size
    for k, cfg in enumerate(arch_grid, start=1):
        hidden_dims = cfg["hidden_dims"]
        dropout = float(cfg.get("dropout", 0.0))
        activation = cfg.get("activation", "relu")
        hidden_dims_str = str(hidden_dims)

        if verbose:
            print("\n" + "=" * 110)
            print(f"[{_ts()}] [{k}/{len(arch_grid)}] ARCH | hidden_dims={hidden_dims} | dropout={dropout} | act={activation} | seed={seed}")
            print("=" * 110)

        for ws in window_sizes:
            # Esperadas: delta_60 x 2 splits (valid/test) = 2 filas (por seed y config)
            df_ws = df_all[
                (df_all["model"] == "mlp") &
                (df_all["target"] == "delta_60") &
                (df_all["window_size"] == ws) &
                (df_all["seed"] == seed) &
                (df_all["hidden_dims"] == hidden_dims_str) &
                (df_all["dropout"] == dropout) &
                (df_all["activation"] == activation) &
                (df_all["lr"] == lr) &
                (df_all["w_decay"] == weight_decay)
            ]

            if len(df_ws) >= 2:
                if verbose:
                    print(f"[SKIP] L{ws}: ya existen valid+test para esta config (seed={seed}).")
                continue

            if verbose:
                print("\n" + "-" * 90)
                print(
                    f"[RUN] target=delta_60 | seed={seed} | L{ws} | in_dim={ws*36} | "
                    f"hidden_dims={hidden_dims} | dropout={dropout} | act={activation} | lr={lr} | w_decay={weight_decay}"
                )
                print("-" * 90)

            # Entrenar/evaluar ESTE ws con ESTA arquitectura
            df_new = run_mlp_delta_60(
                ws,
                seed=seed,
                hidden_dims=hidden_dims,
                dropout=dropout,
                activation=activation,
                lr=lr,
                weight_decay=weight_decay,
                verbose=verbose,
            ).copy()

            # Anti-duplicados por key (usando set incremental)
            new_keys = [tuple(row) for row in df_new[key_cols].values]
            mask_keep = [k_ not in existing_keys for k_ in new_keys]
            df_new = df_new.loc[mask_keep].copy()

            if df_new.empty:
                if verbose:
                    print(f"[INFO] L{ws}: no hubo filas nuevas para agregar.")
            else:
                # Merge + dedupe
                df_all = pd.concat([df_all, df_new], ignore_index=True)
                df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

                # actualizar set de keys
                for k_ in new_keys:
                    existing_keys.add(k_)

                # Guardar SIEMPRE al mismo archivo
                save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

                if verbose:
                    print(f"[OK] Guardado incremental (1 archivo). Total rows={len(df_all)}")

            # liberar memoria por iteración
            del df_new
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return df_all


In [ ]:
window_sizes = [180]

df_all = run_mlp_incremental_delta_60_onefile(
    window_sizes,
    ARCH_GRID,
    seed=42,                # <-- obligatorio
    lr=1e-3,
    weight_decay=1e-4,
    name="mlp_arch_tuning_delta_60_all",
    base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose=True,
)


[13:59:30] [1/17] ARCH | hidden_dims=[64] | dropout=0.0 | act=relu | seed=42

------------------------------------------------------------------------------------------
[RUN] target=delta_60 | seed=42 | L180 | in_dim=6480 | hidden_dims=[64] | dropout=0.0 | act=relu | lr=0.001 | w_decay=0.0001
------------------------------------------------------------------------------------------

[13:59:30] MLP | SEQ2ONE | seed=42 | target=delta_60 | L180 | in_dim=6480 | hidden_dims=[64] | dropout=0.0 | act=relu | lr=0.001 | w_decay=0.0001 | batch=16384 | max_epochs=30 | patience=5
[13:59:30]   [BUILD] Creando bundle (flatten_X=True) ...
H60 Train: (327972, 6480) (327972,)
H60 Valid: (70228, 6480) (70228,)
H60 Test : (70590, 6480) (70590,)
Scaler H60: StandardScaler
[13:59:46]   [BUILD] OK | train X=(327972, 6480) y=(327972,) | dt=15.67s
[13:59:46]   [LOADERS] Creando DataLoaders ...
[13:59:57]   [LOADERS] OK | n(train/valid/test)=(327972/70228/70590) | dt=10.88s
[13:59:57]   [TRAIN] Iniciando entr

## **14. Tuning de Aprendizaje**

In [ ]:
df_mlp_tuning_arch_delta_60 = df_all.copy()
df_mlp_tuning_arch_delta_60